In [1]:
import os
import re
import json
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")

import os, re, json, time
from pathlib import Path

import pandas as pd
import requests
from anthropic import Anthropic

DOMAINS = ["A","B","C","D","E"]

OUTPUT_CONTRACT_DEFAULT = """
Return ONLY valid JSON (no markdown, no commentary).

Schema:
{
  "severity": <int or null>,
  "confidence": <float 0..1 or null>,
  "findings": <list of strings>
}

Rules:
- findings must be short factual phrases grounded in the call text.
- if no findings are available, return [].
- do NOT invent findings.
""".strip()


def _safe_model_name(s: str) -> str:
    return re.sub(r"[^\w\.-]+", "-", s)

def _temp_tag(t: float) -> str:
    if float(t).is_integer():
        return f"t{int(t)}"
    return f"t{str(t).replace('.', '_')}"

def next_free_run_path(results_dir: Path, model: str, temperature: float) -> Path:
    results_dir.mkdir(parents=True, exist_ok=True)
    nums = []
    for p in results_dir.glob("run_*.csv"):
        m = re.search(r"run_(\d+)_", p.name)
        if m:
            nums.append(int(m.group(1)))
    n = (max(nums) + 1) if nums else 1
    return results_dir / f"run_{n}_{_safe_model_name(model)}_{_temp_tag(temperature)}.csv"


def _extract_text_from_anthropic_message(message_obj: dict) -> str:
    # message_obj["content"] ist i.d.R. eine Liste wie [{"type":"text","text":"..."}]
    parts = []
    for c in message_obj.get("content", []) or []:
        if c.get("type") == "text" and "text" in c:
            parts.append(c["text"])
    return "\n".join(parts).strip()


def parse_anthropic_batch_results_jsonl_to_df(jsonl_text: str) -> pd.DataFrame:
    by_call = {}

    for line in jsonl_text.splitlines():
        if not line.strip():
            continue
        rec = json.loads(line)

        custom_id = rec.get("custom_id")  # Notruf_12__A
        if not custom_id or "__" not in custom_id:
            continue
        call_id, domain = custom_id.split("__", 1)
        domain = domain.upper()
        if domain not in DOMAINS:
            continue

        by_call.setdefault(call_id, {"file": call_id})

        result = rec.get("result") or {}
        rtype = result.get("type")

        if rtype != "succeeded":
            # errored / canceled / expired
            by_call[call_id][f"severity_{domain}"] = None
            by_call[call_id][f"confidence_{domain}"] = None
            by_call[call_id][f"findings_{domain}"] = json.dumps([], ensure_ascii=False)
            by_call[call_id][f"error_{domain}"] = json.dumps(result, ensure_ascii=False)
            continue

        msg = result.get("message") or {}
        text = _extract_text_from_anthropic_message(msg)

        # JSON aus Text extrahieren (dein Contract fordert reines JSON, aber wir sind robust)
        m = re.search(r"\{.*\}", text, flags=re.S)
        if not m:
            by_call[call_id][f"severity_{domain}"] = None
            by_call[call_id][f"confidence_{domain}"] = None
            by_call[call_id][f"findings_{domain}"] = json.dumps([], ensure_ascii=False)
            by_call[call_id][f"error_{domain}"] = f"parse_error: no_json_in_text: {text[:200]}"
            continue

        try:
            data = json.loads(m.group(0))
            sev = data.get("severity")
            conf = data.get("confidence")
            findings = data.get("findings", [])
            if findings is None:
                findings = []
            if not isinstance(findings, list):
                findings = [str(findings)]
        except Exception as e:
            by_call[call_id][f"severity_{domain}"] = None
            by_call[call_id][f"confidence_{domain}"] = None
            by_call[call_id][f"findings_{domain}"] = json.dumps([], ensure_ascii=False)
            by_call[call_id][f"error_{domain}"] = f"parse_error: {e}"
            continue

        by_call[call_id][f"severity_{domain}"] = sev
        by_call[call_id][f"confidence_{domain}"] = conf
        by_call[call_id][f"findings_{domain}"] = json.dumps(findings, ensure_ascii=False)
        by_call[call_id][f"error_{domain}"] = None

    df = pd.DataFrame(list(by_call.values()))
    df["call_nr"] = df["file"].str.extract(r"(\d+)").astype("Int64")
    df = df.sort_values(["call_nr", "file"], na_position="last").reset_index(drop=True)

    cols = ["file", "call_nr"]
    for d in DOMAINS:
        cols += [f"severity_{d}", f"confidence_{d}", f"findings_{d}", f"error_{d}"]
    for c in cols:
        if c not in df.columns:
            df[c] = None
    return df[cols]


def run_abcde_batch_experiment_anthropic(
    model: str,
    temperature: float,
    *,
    calls_dir: Path,
    guidelines_dir: Path,
    results_dir: Path,
    max_tokens: int = 400,
    poll_every_seconds: int = 30,
):
    """
    Claude Message Batches:
    - create batch with inline requests
    - poll processing_status until 'ended'
    - download results from results_url (JSONL)
    - parse to DF + save CSV as run_N_<model>_tX.csv
    """
    api_key = os.getenv("ANTHROPIC_API_KEY")
    if not api_key:
        raise EnvironmentError("ANTHROPIC_API_KEY fehlt in env.")
    client = Anthropic(api_key=api_key)

    calls_dir = Path(calls_dir).resolve()
    guidelines_dir = Path(guidelines_dir).resolve()
    results_dir = Path(results_dir).resolve()
    results_dir.mkdir(parents=True, exist_ok=True)

    call_files = sorted(list(calls_dir.glob("*.md")) + list(calls_dir.glob("*.txt")))
    if not call_files:
        raise FileNotFoundError(f"Keine Notrufe gefunden in {calls_dir}")

    # guidelines laden
    domain_guidelines = {}
    for d in DOMAINS:
        p = guidelines_dir / f"{d.lower()}_guidelines.md"
        if not p.exists():
            raise FileNotFoundError(f"Guideline fehlt: {p}")
        domain_guidelines[d] = p.read_text(encoding="utf-8").strip()

    # Batch requests bauen (inline) :contentReference[oaicite:7]{index=7}
    batch_requests = []
    for fp in call_files:
        transcript = fp.read_text(encoding="utf-8").strip()
        for domain in DOMAINS:
            batch_requests.append({
                "custom_id": f"{fp.stem}__{domain}",
                "params": {
                    "model": model,
                    "temperature": temperature,
                    "max_tokens": max_tokens,
                    "system": f"{domain_guidelines[domain]}\n\n{OUTPUT_CONTRACT_DEFAULT}",
                    "messages": [
                        {"role": "user", "content": transcript}
                    ],
                }
            })

    message_batch = client.messages.batches.create(requests=batch_requests)

    # warten bis ended :contentReference[oaicite:8]{index=8}
    while True:
        mb = client.messages.batches.retrieve(message_batch.id)
        if mb.processing_status == "ended":
            message_batch = mb
            break
        print(f"[{message_batch.id}] processing_status={mb.processing_status} ...")
        time.sleep(poll_every_seconds)

    if not message_batch.results_url:
        raise RuntimeError(f"Batch ended, aber results_url fehlt: {message_batch}")

    # results_url downloaden (JSONL) :contentReference[oaicite:9]{index=9}
    headers = {
        "x-api-key": api_key,
        "anthropic-version": "2023-06-01",
    }
    r = requests.get(message_batch.results_url, headers=headers, timeout=300)
    r.raise_for_status()
    results_text = r.text

    # speichern
    out_jsonl = results_dir / f"batch_output_{message_batch.id}.jsonl"
    out_jsonl.write_text(results_text, encoding="utf-8")

    df = parse_anthropic_batch_results_jsonl_to_df(results_text)
    out_csv = next_free_run_path(results_dir, model=model, temperature=temperature)
    df.to_csv(out_csv, index=False, encoding="utf-8")

    return {
        "batch_id": message_batch.id,
        "processing_status": message_batch.processing_status,
        "results_url": message_batch.results_url,
        "output_jsonl_path": str(out_jsonl),
        "csv_path": str(out_csv),
        "df": df,
    }



In [4]:
from pathlib import Path

res = run_abcde_batch_experiment_anthropic(
    model="claude-sonnet-4-6",   # Sonnet 4.x Modellname :contentReference[oaicite:10]{index=10}
    temperature=0.0,
    calls_dir=Path("../../emergency_calls/calls_german"),
    guidelines_dir=Path("guidelines"),
    results_dir=Path("results"),
    max_tokens=400,
)

print(res["csv_path"])
res["df"].head()

[msgbatch_01G4Khc6yhmoyUhZBmTAjsVD] processing_status=in_progress ...
[msgbatch_01G4Khc6yhmoyUhZBmTAjsVD] processing_status=in_progress ...
[msgbatch_01G4Khc6yhmoyUhZBmTAjsVD] processing_status=in_progress ...
[msgbatch_01G4Khc6yhmoyUhZBmTAjsVD] processing_status=in_progress ...
[msgbatch_01G4Khc6yhmoyUhZBmTAjsVD] processing_status=in_progress ...
[msgbatch_01G4Khc6yhmoyUhZBmTAjsVD] processing_status=in_progress ...
[msgbatch_01G4Khc6yhmoyUhZBmTAjsVD] processing_status=in_progress ...
[msgbatch_01G4Khc6yhmoyUhZBmTAjsVD] processing_status=in_progress ...
[msgbatch_01G4Khc6yhmoyUhZBmTAjsVD] processing_status=in_progress ...
[msgbatch_01G4Khc6yhmoyUhZBmTAjsVD] processing_status=in_progress ...
[msgbatch_01G4Khc6yhmoyUhZBmTAjsVD] processing_status=in_progress ...
[msgbatch_01G4Khc6yhmoyUhZBmTAjsVD] processing_status=in_progress ...
[msgbatch_01G4Khc6yhmoyUhZBmTAjsVD] processing_status=in_progress ...
[msgbatch_01G4Khc6yhmoyUhZBmTAjsVD] processing_status=in_progress ...
/Users/s.franke/Deve

,file,call_nr,severity_A,confidence_A,findings_A,error_A,severity_B,confidence_B,findings_B,error_B,...,findings_C,error_C,severity_D,confidence_D,findings_D,error_D,severity_E,confidence_E,findings_E,error_E
0,Notruf_1,1,1,0.5,"[""Patient is speaking coherently throughout th...",None,2,0.7,"[""Patient reports difficulty breathing ('ich k...",None,...,"[""Patient denies circulatory problems when dir...",None,1,0.60,"[""Patient is conscious and able to communicate...",None,1,0.4,"[""Patient is indoors at home (first floor of a...",None
1,Notruf_2,2,1,0.8,"[""Patient is responsive and communicates with ...",None,1,0.9,"[""Caller explicitly states father has no breat...",None,...,"[""Patient is conscious and responsive"", ""No pa...",None,2,0.82,"[""slurred/garbled speech for approximately one...",None,1,0.5,"[""Patient is indoors in a residential apartmen...",None
2,Notruf_3,3,2,0.5,"[""Patient is intoxicated with alcohol (strong ...",None,1,0.7,"[""Caller explicitly states no breathing proble...",None,...,"[""Patient is responsive when spoken to"", ""No r...",None,2,0.60,"[""Patient lying on the street, mostly sleeping...",None,2,0.6,"[""Patient lying on the street in public (outdo...",None
3,Notruf_4,4,1,0.5,"[""Caller reports difficulty breathing ('krieg ...",None,2,0.6,"[""Patient reports difficulty breathing ('krieg...",None,...,"[""Caller denies dizziness when asked about cir...",None,1,0.70,"[""Patient is conscious and communicating coher...",None,1,0.4,"[""Patient is located outdoors on the street in...",None
4,Notruf_5,5,1,0.6,"[""Patient is conscious and screaming, indicati...",None,1,0.5,"[""Caller reports wife occasionally 'panting' (...",None,...,"[""Patient is conscious and responsive (screami...",None,1,0.70,"[""Patient is conscious and responsive (screami...",None,1,0.5,"[""Patient is indoors (private house), no hazar...",None


In [5]:
print(res)

{'status': 'validating', 'batch_id': 'batch_69a1a1ca68dc8190a850b245c3ede1de', 'batch_input_path': '/Users/s.franke/Development/master_clean/experiments/experiment_3/results/batch_input_abcde.jsonl', 'output_jsonl_path': None, 'csv_path': None, 'df': None}
